# QM9 Hidden-Feature Quarto Feasibility Experiment

This independent experiment reads the original QM9 XYZ archive and does not import or modify the existing Quarto engine or benchmark. It never starts a game or selects a hidden mapping. All generated files remain in this directory.

The notebook uses a compact, explicitly defined SMARTS/graph feature registry. Logical implication constrains quartet validity; it does not merge feature identities. Statistical association alone does not exclude a feature.

## 1. Environment and dependency checks

In [1]:
from pathlib import Path
import importlib.util
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rdkit

CWD = Path.cwd()
EXPERIMENT_DIR = CWD if CWD.name == "qm9_quarto" else CWD / "experiments" / "qm9_quarto"
assert (EXPERIMENT_DIR / "qm9_experiment.py").is_file(), EXPERIMENT_DIR
sys.path.insert(0, str(EXPERIMENT_DIR))
import qm9_experiment as experiment

versions = {
    "python": sys.version.split()[0],
    "rdkit": rdkit.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": plt.matplotlib.__version__,
}
assert all(importlib.util.find_spec(name) is not None for name in ("rdkit", "numpy", "pandas", "matplotlib"))
versions

{'python': '3.14.0',
 'rdkit': '2026.09.1',
 'numpy': '2.5.4',
 'pandas': '3.1.1',
 'matplotlib': '3.11.2'}

## 2. Dataset provenance and loading

The analysis streams the source tar.bz2 once, normalizes each source SMILES using RDKit, records parse/consistency exclusions, and writes the deterministic matrix, manifest, registry, provenance, and validation outputs under `experiments/qm9_quarto/`. The public seed creates uniform samples without replacement; failed samples are not replaced.

In [ ]:
results = experiment.run_experiment()
manifest = pd.read_csv(EXPERIMENT_DIR / "entity_manifest.csv")
feature_matrix = pd.read_csv(EXPERIMENT_DIR / "feature_matrix.csv")
feature_registry = pd.read_csv(EXPERIMENT_DIR / "feature_registry.csv")
provenance = json.loads((EXPERIMENT_DIR / "dataset_provenance.json").read_text(encoding="utf-8"))
pd.Series({
    "source": provenance["source_name"],
    "archive": provenance["original_archive_filename"],
    "archive_bytes": provenance["archive_size_bytes"],
    "archive_sha256": provenance["archive_sha256"],
    "source_records": provenance["archive_record_count"],
    "normalized_entities": provenance["parsed_entity_count"],
    "excluded_records": provenance["excluded_record_count"],
})

## 3. Entity normalization

In [ ]:
assert manifest["source_id"].is_unique
assert manifest["parse_status"].eq("parsed_and_sanitized").all()
assert manifest[["original_smiles", "original_inchi", "canonical_isomeric_smiles", "molecular_formula"]].notna().all().all()
assert manifest["source_id"].duplicated().sum() == 0
print("Repeated canonical structures:", provenance["repeated_canonical_structure_count"])
print("Non-isomeric graphs with multiple isomeric SMILES:", provenance["nonisomeric_graphs_with_multiple_isomeric_smiles"])
manifest[["source_id", "original_smiles", "original_inchi", "canonical_isomeric_smiles", "molecular_formula", "heavy_atom_count", "formal_charge", "explicit_hydrogen_graph_nodes", "implicit_hydrogen_count", "parse_status"]].head(8)

## 4. Scientific feature registry

Scientifically valid constant features remain in the registry but are excluded from the gameplay candidate pool. Feature definitions include graph predicates, RDKit version, hydrogen semantics, dependencies, population coverage, and gameplay eligibility.

In [ ]:
assert feature_registry["feature_id"].is_unique
assert feature_registry["scientifically_valid"].all()
assert feature_registry["missing_count"].eq(0).all()
feature_registry[["feature_id", "family", "definition", "graph_predicate", "hydrogen_semantics", "dependencies", "true_count", "prevalence", "constant_in_population", "eligible_for_gameplay"]]

## 5. Feature matrix construction and integrity checks

In [ ]:
feature_ids = feature_registry["feature_id"].tolist()
assert feature_matrix["source_id"].is_unique
assert feature_matrix[feature_ids].notna().all().all()
assert all(set(feature_matrix[feature_id].unique()).issubset({True, False}) for feature_id in feature_ids)
assert feature_matrix["source_id"].tolist() == manifest["source_id"].tolist()
print("Matrix shape:", feature_matrix.shape)
print("Gameplay candidates:", results["gameplay_feature_ids"])
feature_matrix.head(8)

## 6. Feature relationship analysis

Exact semantic duplicates, dataset-level duplicates, complements, implications, and phi associations are reported separately. An implication is a validity constraint, not feature equivalence. Correlation is descriptive only.

In [ ]:
relations = results["feature_relationships"]
relationship_counts = pd.Series({
    "exact semantic duplicate groups": len(relations["exact_semantic_duplicates"]),
    "dataset exact duplicate pairs": len(relations["dataset_exact_duplicates"]),
    "dataset complement pairs": len(relations["dataset_complements"]),
    "full-dataset empirical implications": len(relations["full_dataset_implications"]),
    "pairwise associations": len(relations["pairwise_phi_associations"]),
})
display(relationship_counts)
display(pd.DataFrame(relations["dataset_exact_duplicates"]))
display(pd.DataFrame(relations["dataset_complements"]))
display(pd.DataFrame(relations["full_dataset_implications"]))
pd.DataFrame(relations["pairwise_phi_associations"]).assign(abs_phi=lambda x: x["phi"].abs()).sort_values("abs_phi", ascending=False).head(20)

## 7. Quartet feasibility

A valid four-feature set must induce exactly the 16 distinct signatures on a selected 16-molecule sample. Game-structural classes canonicalize entity-to-signature mappings under feature permutation and independent global bit complementation; implication-related feature identities are never merged.

In [ ]:
sample_summary = results["sample_summary"]
pd.Series(sample_summary)

quartet_counts = pd.Series(sample_summary["raw_valid_quartet_count_distribution"], dtype="int64")
quartet_counts.index = quartet_counts.index.astype(int)
quartet_counts.sort_index().rename_axis("valid quartets per sample").to_frame("sample count")

## 8. Reproducible random sampling

All samples come from the successfully normalized QM9 population by uniform sampling without replacement. Failed samples are retained as failures and never replaced with constructive samples.

In [ ]:
assert results["sample_design"]["sample_count"] == 1024
assert results["sample_design"]["replacement_of_failed_samples"] is False
sample_rows = pd.DataFrame([
    {
        "sample_index": sample["sample_index"],
        "valid_semantic_quartets": sample["raw_valid_semantic_quartets"],
        "game_structural_quartets": sample["game_structurally_distinct_valid_quartets"],
        "valid": sample["raw_valid_semantic_quartets"] > 0,
    }
    for sample in results["samples"]
])
display(sample_rows.head())
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sample_rows["valid_semantic_quartets"], bins=30, color="#3b6ea8", edgecolor="white")
ax.set(title="Valid semantic quartets across 1,024 random samples", xlabel="Valid quartets", ylabel="Samples")
plt.show()
sample_rows.describe(include="all")

## 9. Draft-stage ambiguity

For every extendable, nonterminal accepted-feature prefix represented by the specified ownership model, the notebook counts consistent final semantic quartets and game-structural classes separately. Raw counts are not entropy. No ambiguity floor is imposed.

In [ ]:
draft_summary = results["draft_summary"]
display(pd.Series(draft_summary))
semantic_states = pd.Series(draft_summary["semantic_state_count_distribution"], dtype="int64")
structural_states = pd.Series(draft_summary["structural_state_count_distribution"], dtype="int64")
fig, ax = plt.subplots(figsize=(9, 4))
if not semantic_states.empty:
    semantic_states.index = semantic_states.index.astype(int)
    semantic_states.sort_index().plot.bar(ax=ax, color="#4a8f69")
ax.set(title="Remaining semantic hypotheses over extendable draft prefixes", xlabel="Consistent final quartets", ylabel="Prefix states")
plt.show()
pd.DataFrame({"semantic_prefix_states": semantic_states, "structural_prefix_states": structural_states})

## 10. Results and limitations

QM9 is a computationally generated molecular dataset with a deliberately restricted chemical domain. SMARTS predicates are operational and may be narrower than broad chemical class names. Empirical full-dataset or sample implications are not automatically universal chemical laws. The Figshare archive's exact reuse license and redistribution terms must be verified before redistributing the raw archive.

In [ ]:
print("Scientific features:", len(feature_registry))
print("Gameplay-eligible features:", len(results["gameplay_feature_ids"]))
print("Samples with valid quartets:", sample_summary["samples_with_valid_quartet"], "/", sample_summary["sample_count"])
print("Draft prefix states analyzed:", draft_summary["all_prefix_states"])
print("No game started; no hidden entity-to-piece mapping was created.")
print("License status:", provenance["source_license_note"])
pd.DataFrame(results["scientific_limitations"], columns=["unresolved limitation"])

## 11. Reproducibility summary

Re-running this notebook downloads nothing when the preserved archive is present. The SHA-256, feature registry order, sampling seed, sample size, and sample count are written to local provenance/configuration outputs. All outputs are confined to `experiments/qm9_quarto/`.

In [ ]:
config = json.loads((EXPERIMENT_DIR / "preprocessing_config.json").read_text(encoding="utf-8"))
assert config["sample_count"] == 1024 and config["sample_size"] == 16
assert config["public_sample_seed"] == results["sample_design"]["seed"]
assert (EXPERIMENT_DIR / "results.json").is_file()
assert (EXPERIMENT_DIR / "feature_matrix.csv").is_file()
assert (EXPERIMENT_DIR / "entity_manifest.csv").is_file()
config